# Hotel Booking Cancellation Prediction
## End-to-End Machine Learning Pipeline & Clean EDA

### Business Objective
Predict whether a hotel booking will be canceled before arrival. Accurate predictions allow revenue managers and front-desk teams to:
1. Implement proactive guest retention or engagement campaigns.
2. Dynamically optimize overbooking rates to minimize vacant rooms without incurring walk costs.
3. Mitigate lost revenue from last-minute cancellations.

## 1. Setup & Environment
Import verified libraries for data processing, feature engineering, modeling, and evaluation.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-learn tools
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

# Visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

## 2. Data Ingestion & Quality Diagnostics
Load the dataset from the local archive and perform an initial diagnostic audit.

In [ ]:
data_path = "../data/hotel_bookings.csv.zip"
if not os.path.exists(data_path):
    data_path = "../data/hotel_bookings.csv"

df = pd.read_csv(data_path)
print(f"Raw dataset dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())

In [ ]:
# Target Distribution Check
target_counts = df["is_canceled"].value_counts()
target_pct = df["is_canceled"].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({"Count": target_counts, "Percentage (%)": target_pct.round(2)})
dist_df.index = ["Not Canceled (0)", "Canceled (1)"]
display(dist_df)

In [ ]:
# Missing Value Audit
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_summary = pd.DataFrame({
    "Missing Values": missing,
    "Percentage (%)": (missing / len(df) * 100).round(2)
})
print("Columns with missing values:")
display(missing_summary)

In [ ]:
# Duplicate Row Analysis
exact_duplicates = df.duplicated().sum()
print(f"Exact duplicate rows in dataset: {exact_duplicates:,} ({exact_duplicates / len(df) * 100:.1f}% of total)")
print("Removing exact duplicates is essential to prevent data leakage between train and test splits.")

## 3. Exploratory Data Analysis & Anomaly Discovery
Key findings from data exploration:
1. **Average Daily Rate (ADR)**: contains negative values (e.g. -6.38) and extreme outliers (5,400).
2. **Guest counts**: records with 0 adults, 0 children, and 0 babies exist.
3. **Agent & Company columns**: numeric ID codes stored as floats due to nulls; must be treated as categorical identifiers.

In [ ]:
# ADR Outlier & Negative Value Inspection
print("ADR summary statistics:")
print(df["adr"].describe().round(2))
print(f"\nNegative ADR bookings count: {(df['adr'] < 0).sum()}")
print(f"Extreme ADR (> $1,000) bookings count: {(df['adr'] > 1000).sum()}")

In [ ]:
# Zero Guest Booking Audit
zero_guests = (df["adults"] == 0) & (df["children"] == 0) & (df["babies"] == 0)
print(f"Bookings with zero total guests: {zero_guests.sum():,}")

## 4. Data Cleaning & Leakage Prevention
We apply strict data hygiene:
- Remove exact duplicate rows.
- Replace negative ADR with NaN for median imputation; cap extreme ADR outliers.
- Format `agent` and `company` as categorical strings with 'Unknown' for missing.
- Flag zero-guest bookings.
- **Drop post-booking leakage columns**: `reservation_status`, `reservation_status_date`, `assigned_room_type`, `booking_changes`.

In [ ]:
data = df.copy()

# 1. Deduplicate
data = data.drop_duplicates().reset_index(drop=True)

# 2. Sanitize ADR
data.loc[data["adr"] < 0, "adr"] = np.nan
data.loc[data["adr"] > 5000, "adr"] = 5000.0

# 3. Treat agent and company as categorical IDs
for col in ["agent", "company"]:
    data[col] = data[col].astype("Int64").astype("string").fillna("Unknown").astype(object)

# 4. Impute standard missing
data["country"] = data["country"].fillna("Unknown")
data["children"] = data["children"].fillna(0.0)

# 5. Create zero guest flag
data["zero_guest_booking"] = (
    (data["adults"] == 0) & (data["children"] == 0) & (data["babies"] == 0)
).astype(int)

# 6. Remove target leakage columns
leakage_columns = [
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type",
    "booking_changes"
]
data = data.drop(columns=[col for col in leakage_columns if col in data.columns])

print(f"Cleaned dataset dimensions: {data.shape}")

## 5. Domain Feature Engineering
Create features to capture travel party structure, booking agency relationships, and guest behavior.

In [ ]:
# Derived domain features
data["total_stay_nights"] = data["stays_in_weekend_nights"] + data["stays_in_week_nights"]
data["total_guests"] = data["adults"] + data["children"] + data["babies"]
data["is_alone"] = (data["total_guests"] == 1).astype(int)
data["has_children_or_babies"] = ((data["children"] > 0) | (data["babies"] > 0)).astype(int)
data["is_agent_booking"] = (data["agent"] != "Unknown").astype(int)
data["is_company_booking"] = (data["company"] != "Unknown").astype(int)
data["has_previous_cancellations"] = (data["previous_cancellations"] > 0).astype(int)
data["has_special_requests"] = (data["total_of_special_requests"] > 0).astype(int)
data["has_parking_space"] = (data["required_car_parking_spaces"] > 0).astype(int)

print("Engineered feature sample:")
display(data[["total_stay_nights", "total_guests", "is_alone", "has_special_requests", "has_previous_cancellations"]].head())

## 6. Leak-Free Train / Validation / Test Splitting
We hold out 20% for final model testing. The remaining 80% is partitioned into:
- **Train partition (64% of total)**: to fit models and transformers.
- **Validation partition (16% of total)**: strictly to tune the decision threshold without data contamination.

In [ ]:
X = data.drop(columns=["is_canceled"])
y = data["is_canceled"].astype(int)

# Initial train-test split (80% dev, 20% test)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Validation partition from dev set (80% train_fit, 20% val)
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_dev, y_dev, test_size=0.2, random_state=42, stratify=y_dev
)

print(f"Train fit partition : {X_train_fit.shape[0]:,} samples")
print(f"Validation partition: {X_val.shape[0]:,} samples")
print(f"Held-out Test set   : {X_test.shape[0]:,} samples")

## 7. Pipeline Construction & Model Comparison
Build Scikit-Learn `ColumnTransformer` with infrequent category handling to prevent high-cardinality explosion from agents and countries.

In [ ]:
numeric_features = X_train_fit.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train_fit.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

try:
    ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01, sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore")

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", ohe)
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=20, min_samples_split=5, min_samples_leaf=2,
        class_weight="balanced", random_state=42, n_jobs=-1
    )
}

In [ ]:
trained_pipelines = {}
model_results = []

for name, clf in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", clf)])
    pipe.fit(X_train_fit, y_train_fit)
    trained_pipelines[name] = pipe
    
    val_probs = pipe.predict_proba(X_val)[:, 1]
    val_preds = pipe.predict(X_val)
    
    model_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, val_preds),
        "Precision": precision_score(y_val, val_preds),
        "Recall": recall_score(y_val, val_preds),
        "F1-Score": f1_score(y_val, val_preds),
        "ROC-AUC": roc_auc_score(y_val, val_probs)
    })

display(pd.DataFrame(model_results).round(4))

## 8. Validation Decision Threshold Optimization
By default, classifiers predict positive when probability >= 0.5. In cancellation prediction, false negatives (missed cancellations) vs. false positives (unnecessary interventions) have distinct costs.
We sweep thresholds on the strictly held-out validation set to optimize F1-score.

In [ ]:
rf_pipe = trained_pipelines["Random Forest"]
rf_val_probs = rf_pipe.predict_proba(X_val)[:, 1]

threshold_rows = []
thresholds = np.arange(0.20, 0.85, 0.05)

for t in thresholds:
    preds = (rf_val_probs >= t).astype(int)
    threshold_rows.append({
        "Threshold": round(t, 2),
        "Precision": precision_score(y_val, preds, zero_division=0),
        "Recall": recall_score(y_val, preds, zero_division=0),
        "F1-Score": f1_score(y_val, preds, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)
best_idx = threshold_df["F1-Score"].idxmax()
best_threshold = float(threshold_df.loc[best_idx, "Threshold"])

print(f"Optimal Decision Threshold: {best_threshold} (Validation F1 = {threshold_df.loc[best_idx, 'F1-Score']:.4f})")

# Plot Precision, Recall, and F1 curve
plt.figure(figsize=(9, 5))
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], label="Precision", marker="o")
plt.plot(threshold_df["Threshold"], threshold_df["Recall"], label="Recall", marker="s")
plt.plot(threshold_df["Threshold"], threshold_df["F1-Score"], label="F1-Score", marker="^", lw=2.5)
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Optimal Threshold ({best_threshold})")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Precision, Recall, and F1 vs. Decision Threshold")
plt.legend()
plt.show()

## 9. Final Held-Out Test Evaluation
Evaluate the Random Forest pipeline with calibrated threshold on the held-out test partition.

In [ ]:
test_probs = rf_pipe.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

print("=================== Final Test Set Evaluation ===================")
print(f"Accuracy : {accuracy_score(y_test, test_preds):.4f}")
print(f"Precision: {precision_score(y_test, test_preds):.4f}")
print(f"Recall   : {recall_score(y_test, test_preds):.4f}")
print(f"F1-Score : {f1_score(y_test, test_preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, test_probs):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, test_probs):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=["Not Canceled", "Canceled"]))

# Confusion Matrix
cm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Not Canceled", "Canceled"], yticklabels=["Not Canceled", "Canceled"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Test Set)")
plt.show()

## 10. Feature Importance Analysis
Identify the most predictive signals governing booking cancellations.

In [ ]:
fitted_prep = rf_pipe.named_steps["preprocessor"]
fitted_rf = rf_pipe.named_steps["model"]
feat_names = fitted_prep.get_feature_names_out()

feat_imp = pd.DataFrame({
    "Feature": feat_names,
    "Importance": fitted_rf.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp.head(15), x="Importance", y="Feature", palette="viridis")
plt.title("Top 15 Most Important Features in Cancellation Prediction")
plt.xlabel("Gini Importance")
plt.show()

## 11. Model Bundle Serialization & Production Inference
Save the complete production bundle containing the pipeline, calibrated threshold, expected schema, and metadata.

In [ ]:
model_bundle = {
    "model": rf_pipe,
    "threshold": float(best_threshold),
    "feature_columns": X.columns.tolist(),
    "target": "is_canceled"
}

os.makedirs("../models", exist_ok=True)
model_save_path = "../models/hotel_cancellation_model.joblib"
joblib.dump(model_bundle, model_save_path)
print(f"Production model bundle successfully saved to: {model_save_path}")

In [ ]:
# Test Inference on a Sample Booking
loaded_bundle = joblib.load(model_save_path)
loaded_model = loaded_bundle["model"]
decision_thresh = loaded_bundle["threshold"]

sample_booking = X_test.iloc[[0]]
sample_proba = loaded_model.predict_proba(sample_booking)[0, 1]
sample_pred = int(sample_proba >= decision_thresh)

print(f"Sample Booking Cancellation Probability: {sample_proba:.3f}")
print(f"Calibrated Threshold                 : {decision_thresh:.3f}")
print(f"Predicted Outcome                    : {'Cancelled (1)' if sample_pred == 1 else 'Not Cancelled (0)'}")
print(f"Actual Ground Truth                  : {'Cancelled (1)' if y_test.iloc[0] == 1 else 'Not Cancelled (0)'}")